# 🦀 Day 4: Authentication — Password Hashing and JWT

---

## Password Hashing — Never Store Plaintext

**Rule #1:** Never store passwords in plaintext. If your DB is compromised, attackers get everything.

**Solution:** Store a **hash** of the password. Hashing is one-way — you can't recover the original. To verify, hash the input and compare.

**Salt + hash:** Add a random **salt** before hashing to prevent rainbow table attacks. Each user gets a unique salt.

In [ ]:
:dep sha2 = "0.10"
:dep base64 = "0.22"

use sha2::{Sha256, Digest};

fn simple_hash_b64(password: &str, salt: &str) -> String {
    let mut hasher = Sha256::new();
    hasher.update(salt.as_bytes());
    hasher.update(password.as_bytes());
    base64::Engine::encode(&base64::engine::general_purpose::STANDARD, hasher.finalize())
}

let salt = "random_salt_123";
let hash = simple_hash_b64("mypassword", salt);
println!("Hash: {}", hash);

// Verify: same input -> same hash
let hash2 = simple_hash_b64("mypassword", salt);
println!("Match: {}", hash == hash2);

In [ ]:
// simple_hash_b64 is defined above — use it for password verification in your app
println!("Password hashing ready.");

## Production Note

**SHA-256 is not ideal for passwords.** Use **argon2** or **bcrypt** — they're designed to be slow (resist brute force). We use SHA-256 here for simplicity in the notebook.

```rust
// Production: use argon2
argon2::hash_encoded(password, salt, &config)?;
```

## JWT Structure: header.payload.signature

A **JWT** (JSON Web Token) has three base64url-encoded parts:
1. **Header** — algorithm (e.g. HS256)
2. **Payload** — claims (sub, exp, iat, custom data)
3. **Signature** — HMAC-SHA256(header + "." + payload, secret)

Format: `base64(header).base64(payload).base64(signature)`

In [ ]:
:dep base64 = "0.22"
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep hmac = "0.12"
:dep sha2 = "0.10"

use base64::Engine;
use hmac::{Hmac, Mac};
use serde::{Deserialize, Serialize};
use sha2::Sha256;

type HmacSha256 = Hmac<Sha256>;

#[derive(Debug, Serialize, Deserialize)]
struct JwtHeader {
    alg: String,
    typ: String,
}

#[derive(Debug, Serialize, Deserialize)]
struct JwtClaims {
    sub: String,   // subject (user id)
    exp: u64,      // expiration timestamp
    iat: u64,      // issued at timestamp
}

fn b64_encode_json<T: Serialize>(v: &T) -> String {
    let json = serde_json::to_string(v).unwrap();
    base64::engine::general_purpose::URL_SAFE_NO_PAD.encode(json.as_bytes())
}

fn create_jwt(secret: &str, sub: &str, exp_secs: u64) -> String {
    let iat = std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).unwrap().as_secs();
    let exp = iat + exp_secs;
    let header = JwtHeader { alg: "HS256".into(), typ: "JWT".into() };
    let claims = JwtClaims { sub: sub.to_string(), exp, iat };
    let header_b64 = b64_encode_json(&header);
    let payload_b64 = b64_encode_json(&claims);
    let message = format!("{}.{}", header_b64, payload_b64);
    let mut mac = HmacSha256::new_from_slice(secret.as_bytes()).unwrap();
    mac.update(message.as_bytes());
    let sig = mac.finalize().into_bytes();
    let sig_b64 = base64::engine::general_purpose::URL_SAFE_NO_PAD.encode(&sig);
    format!("{}.{}", message, sig_b64)
}

let token = create_jwt("my-secret-key", "user123", 3600);
println!("JWT: {}", token);

## Verifying a JWT

1. Split by `.` to get header, payload, signature
2. Recompute HMAC(header.payload, secret)
3. Compare with provided signature
4. Decode payload and check `exp` (expiration)

In [ ]:
use base64::Engine;
use hmac::{Hmac, Mac};
use serde::Deserialize;

fn verify_jwt(secret: &str, token: &str) -> Result<JwtClaims, String> {
    let parts: Vec<&str> = token.split('.').collect();
    if parts.len() != 3 { return Err("Invalid token format".into()); }
    let (header_payload, sig_b64) = (format!("{}.{}", parts[0], parts[1]), parts[2]);
    let mut mac = HmacSha256::new_from_slice(secret.as_bytes()).map_err(|e| e.to_string())?;
    mac.update(header_payload.as_bytes());
    let expected_sig = base64::engine::general_purpose::URL_SAFE_NO_PAD.encode(&mac.finalize().into_bytes());
    if sig_b64 != expected_sig { return Err("Invalid signature".into()); }
    let payload_bytes = base64::engine::general_purpose::URL_SAFE_NO_PAD.decode(parts[1].as_bytes()).map_err(|e| e.to_string())?;
    let payload_str = String::from_utf8(payload_bytes).map_err(|e| e.to_string())?;
    let claims: JwtClaims = serde_json::from_str(&payload_str).map_err(|e| e.to_string())?;
    let now = std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).unwrap().as_secs();
    if claims.exp < now { return Err("Token expired".into()); }
    Ok(claims)
}

let token = create_jwt("my-secret-key", "user123", 3600);
match verify_jwt("my-secret-key", &token) {
    Ok(claims) => println!("Valid! sub={}, exp={}", claims.sub, claims.exp),
    Err(e) => println!("Invalid: {}", e),
}

// Wrong secret fails
match verify_jwt("wrong-secret", &token) {
    Ok(_) => println!("Unexpected success"),
    Err(e) => println!("Wrong secret correctly rejected: {}", e),
}

## Token Expiration

Always check `exp` (expiration) when verifying. Expired tokens should be rejected.

In [ ]:
// Create a token that expires "now" (exp = iat, so already expired)
// We'd need to tweak create_jwt to accept custom iat/exp. For demo, we'll create with 0 exp.

fn create_jwt_expired(secret: &str, sub: &str) -> String {
    let now = std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).unwrap().as_secs();
    let header = JwtHeader { alg: "HS256".into(), typ: "JWT".into() };
    let claims = JwtClaims { sub: sub.to_string(), exp: now - 1, iat: now - 3600 }; // exp in past
    let header_b64 = b64_encode_json(&header);
    let payload_b64 = b64_encode_json(&claims);
    let message = format!("{}.{}", header_b64, payload_b64);
    let mut mac = HmacSha256::new_from_slice(secret.as_bytes()).unwrap();
    mac.update(message.as_bytes());
    let sig_b64 = base64::engine::general_purpose::URL_SAFE_NO_PAD.encode(&mac.finalize().into_bytes());
    format!("{}.{}", message, sig_b64)
}

let expired = create_jwt_expired("secret", "user1");
match verify_jwt("secret", &expired) {
    Ok(_) => println!("Bug: expired token accepted!"),
    Err(e) => println!("Expired token correctly rejected: {}", e),
}

## 🏋️ Exercise

Add **role-based claims** to the JWT. Extend `JwtClaims` with a `role` field (e.g. `"admin"` or `"user"`). Update `create_jwt` and `verify_jwt` to handle it. Create a token with `role: "admin"` and verify it prints the role.

In [ ]:
:dep base64 = "0.22"
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep hmac = "0.12"
:dep sha2 = "0.10"

// YOUR CODE HERE
// 1. Add role: String to JwtClaims
// 2. Update create_jwt to accept role parameter
// 3. Update verify_jwt (it already decodes claims, so role will be there)
// 4. Create token with role "admin", verify, print claims.role

## 🎯 Key Takeaways

1. **Never store plaintext passwords** — use salt + hash
2. **SHA-256** — simple demo; production use argon2/bcrypt
3. **JWT** — header.payload.signature, base64url-encoded
4. **Claims** — sub (subject), exp (expiration), iat (issued at)
5. **HMAC-SHA256** — sign with secret to prevent tampering
6. **Verification** — check signature and exp before trusting token

---

**Tomorrow:** Environment configuration (env vars, .env, config) ⚙️